# Phase 4: Typewell matching — findings, harness, and a scaffold to iterate

This notebook does three things, in order:
1. **Documents the diagnostic journey** that reframed the problem (so it is not lost).
2. **Provides a correct scoring harness** at the *real* eval difficulty (~73% tail).
3. **Implements four rungs** — floor, geometry backbone, typewell-match correction,
   and an oracle ceiling — with the matcher written as a clearly-labelled
   **scaffold to improve**, not a finished solution.

> **Honesty note.** As of this notebook, the matcher does **not** yet beat the
> floor at the real 73% mask. What *is* established: the problem's true shape, why
> earlier approaches failed, and that a near-perfect solution exists (oracle ≈
> 0.14 ft). The matcher is the open problem; everything around it is solid.

## The diagnostic journey (why the problem looks the way it does)

We arrived here by elimination, each step backed by data on real wells:

**1. The eval zone is a ~73% forward tail, not a short one.**
Test wells hide TVT on the last 67–80% of rows (mean **73%**), contiguous from a
point to the toe. Every method must extrapolate across three-quarters of the well.

**2. Feature regressors lose to a flat floor (notebook 3).**
Predicting per-row ΔTVT and integrating it forward accumulates drift; holding the
last value flat beats it. Conventional tabular regression is the wrong framing.

**3. Absolute geometry can't generalize (notebook 3b).**
`X, Y` are geographic coordinates and `MD` is a row counter (ΔMD≡1). They don't
transfer across wells. Only *relative* quantities (`dZ/dMD`, curvature, GR shape) do.

**4. Lateral and typewell are sampled on different vertical rulers.**
The lateral moves ~**0.01 ft of TVT per row** (it is near-horizontal); the typewell
steps **0.5 ft per row**. A fixed *row-count* window compares a ~0.4-ft lateral
slice against a ~20-ft typewell slice — a ~50× mismatch. Matching by row index is
meaningless; it must be done in **TVT space**. (Row-space true-location correlation
≈ 0.00; TVT-aligned it rises to +0.3–0.5.)

**5. A single GR value cannot localize.**
One GR level occurs at ~**59 different TVT depths** in the typewell. Value-matching
lands anywhere (errors of ~240 ft). Only *pattern/position*, constrained to a
neighborhood, can localize.

**6. The right target is the offset `TVT + Z`, not TVT.**
`TVT + Z` (a horizon-depth proxy) is ~3× smoother than TVT (per-row std 0.075 vs
0.27). Predict the smooth offset, then recover `TVT = offset − Z` using the known
`Z`. This puts the bit's vertical position to work and leaves only a gentle drift
to model.

**7. The backbone must extrapolate; trees cannot.**
A tree predicting absolute offset clamps to its training range and fails on the
tail (28 ft vs an 8 ft polyfit). The geometric backbone must be an extrapolating
fit; any learned model may only predict the *residual*.

**8. At 73%, geometry extrapolation also breaks — so the typewell is essential.**
Over a 73% tail, even a degree-2 offset polyfit degrades (≈12 ft) and degree-3
explodes (≈700 ft). Yet the eval TVT lies *inside* the known TVT range and the
typewell grid passes through it (**oracle ≈ 0.14 ft**). The information is present;
only a TVT-space, position-based typewell match can reach it without extrapolating.

**Conclusion.** The load-bearing signal is **typewell position matching in TVT
space**, predicting the smooth offset, with geometry as a (weak, at 73%) prior.
That is the rung this notebook scaffolds.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")

# The REAL eval fraction, measured from test wells' TVT_input NaN rate.
REAL_EVAL_FRAC = 0.73  # mean 73%, range 67-80%
MASKS = [0.40, 0.60, 0.73]  # include the real value; 0.40/0.60 show the regime trend


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


def tail_mask(n, frac):
    k = round(n * frac)
    m = np.zeros(n, bool)
    if k:
        m[n - k :] = True
    return m

## Load wells (lateral + typewell together)

The matcher needs both per well, so we load them paired. Only train wells have
truth `TVT` for scoring; the typewell (`TVT, GR_z, Geology_canon`) is the
reference each lateral is matched against.

In [2]:
def load_pair(well_id, split="train"):
    hz = (
        pd.read_csv(CLEAN_DIR / split / f"{well_id}__horizontal_well.csv", dtype={"well_id": str})
        .sort_values("MD")
        .reset_index(drop=True)
    )
    tw = pd.read_csv(
        CLEAN_DIR / split / f"{well_id}__typewell.csv", dtype={"well_id": str}
    ).reset_index(drop=True)
    return hz, tw


def list_train_wells():
    return sorted(
        p.name.split("__")[0] for p in (CLEAN_DIR / "train").glob("*__horizontal_well.csv")
    )


WELLS = list_train_wells()
print(f"{len(WELLS)} train wells available")
hz0, tw0 = load_pair(WELLS[0])
print(
    "lateral cols:",
    [
        c
        for c in hz0.columns
        if c in ("MD", "X", "Y", "Z", "GR", "GR_z", "TVT", "TVT_input", "gr_missing")
    ],
)
print("typewell cols:", list(tw0.columns))

773 train wells available
lateral cols: ['MD', 'X', 'Y', 'Z', 'TVT', 'GR', 'TVT_input', 'gr_missing', 'GR_z']
typewell cols: ['well_id', 'TVT', 'GR', 'Geology', 'row_idx', 'GR_raw', 'GR_z', 'Geology_canon']


## Rung 0 — Floor (hold last TVT)

No model. Carries the last known TVT flat across the eval tail. The bar to beat.

In [3]:
def rung_floor(hz, frac):
    tvt = hz["TVT"].values.astype(float)
    m = tail_mask(len(hz), frac)
    last = np.where(~m)[0][-1]
    pred = np.full(m.sum(), tvt[last])
    return pred, tvt[m], m

## Rung 1 — Geometry backbone (offset = TVT + Z, extrapolating polyfit)

Fit `offset = TVT + Z` as a polynomial in MD on the known head, extrapolate across
the tail, recover `TVT = offset − Z`. Degree is a knob: degree-2 is best mid-range,
but **all degrees degrade at 73%** — that degradation is a documented finding, not a
bug. We expose `degree` so the harness can show the regime dependence.

In [4]:
def rung_geometry(hz, frac, degree=2):
    tvt = hz["TVT"].values.astype(float)
    Z = hz["Z"].values.astype(float)
    MD = hz["MD"].values.astype(float)
    offset = tvt + Z
    m = tail_mask(len(hz), frac)
    kn = np.where(~m)[0]
    A = np.polyfit(MD[kn], offset[kn], degree)
    pred = np.polyval(A, MD[m]) - Z[m]
    return pred, tvt[m], m

## Rung 2 — Typewell-match correction  ⚠️ SCAFFOLD — iterate here

This is the open problem. The intended design, from the findings:
- work in **TVT space** (not row index),
- predict the **offset**, constrained to a neighborhood (no 59-way value ambiguity),
- use the typewell GR **pattern/position**, not a single GR value.

The implementation below is a *first scaffold*: for each eval row it searches the
typewell within a TVT window around the geometry prior and picks the position whose
local GR pattern best matches. **It does not yet beat the floor at 73%** — it is
here to be improved (better pattern features, DTW, multi-scale windows, learned
residual correction). The harness scores it honestly alongside the others.

In [5]:
def rung_typewell_match(hz, tw, frac, win_ft=8.0, search_ft=30.0, degree=1):
    # SCAFFOLD matcher. Returns (pred_tvt, true_tvt, mask).
    # Strategy: geometry prior gives expected TVT per eval row; refine by finding the
    # typewell TVT (within +/- search_ft) whose GR pattern best matches the lateral's
    # local GR pattern, both expressed on a shared fine TVT grid.
    tvt = hz["TVT"].values.astype(float)
    Z = hz["Z"].values.astype(float)
    MD = hz["MD"].values.astype(float)
    GRz = hz["GR_z"].values.astype(float)
    offset = tvt + Z
    m = tail_mask(len(hz), frac)
    kn = np.where(~m)[0]
    ev = np.where(m)[0]

    # geometry prior (expected TVT on the tail)
    A = np.polyfit(MD[kn], offset[kn], degree)
    prior_tvt = np.polyval(A, MD) - Z

    twt = tw["TVT"].values.astype(float)
    twg = tw["GR_z"].values.astype(float)
    grid = np.arange(-win_ft, win_ft + 1e-9, 0.5)

    # lateral GR as a function of its (prior) TVT, for building local patterns
    lat_tvt_all = prior_tvt
    order = np.argsort(lat_tvt_all)
    lt, lg = lat_tvt_all[order], GRz[order]
    good = ~np.isnan(lg)
    lt, lg = lt[good], lg[good]

    preds = np.empty(len(ev))
    for r, i in enumerate(ev):
        exp = prior_tvt[i]
        if np.isnan(GRz[i]) or len(lt) < 5:
            preds[r] = exp
            continue
        lat_pat = np.interp(exp + grid, lt, lg)
        if np.std(lat_pat) < 1e-6:
            preds[r] = exp
            continue
        cand = twt[(twt >= exp - search_ft) & (twt <= exp + search_ft)]
        best_c, best_t = -2.0, exp
        for c0 in cand:
            seg = np.interp(c0 + grid, twt, twg)
            if np.std(seg) < 1e-6:
                continue
            cc = np.corrcoef(lat_pat, seg)[0, 1]
            if cc > best_c:
                best_c, best_t = cc, c0
        preds[r] = best_t
    return preds, tvt[ev], m

## Rung 3 — Oracle ceiling (typewell grid resolution)

The best any typewell-based method could do: snap each true eval TVT to the nearest
typewell grid point. Not a predictor (it peeks at truth) — it quantifies the
**headroom**. ≈ 0.14 ft confirms the signal is fully present; the gap to the other
rungs is entirely matcher quality.

In [6]:
def rung_oracle(hz, tw, frac):
    tvt = hz["TVT"].values.astype(float)
    m = tail_mask(len(hz), frac)
    twt = tw["TVT"].values.astype(float)
    snapped = np.array([twt[np.argmin(np.abs(twt - t))] for t in tvt[m]])
    return snapped, tvt[m], m

## Scoring harness

Score every rung at every mask, pooled across wells (RMSE on TVT), plus per-well
diagnostics: the geometry R² (how linear the offset is — low means geometry is
weak and the matcher matters more) and the residual std after geometry.

In [7]:
def geometry_r2(hz, frac, degree=2):
    tvt = hz["TVT"].values.astype(float)
    Z = hz["Z"].values.astype(float)
    MD = hz["MD"].values.astype(float)
    offset = tvt + Z
    kn = np.where(~tail_mask(len(hz), frac))[0]
    A = np.polyfit(MD[kn], offset[kn], degree)
    pr = np.polyval(A, MD[kn])
    ss_res = np.sum((offset[kn] - pr) ** 2)
    ss_tot = np.sum((offset[kn] - offset[kn].mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan


def score_all(wells, masks, n_wells=None):
    wells = wells[:n_wells] if n_wells else wells
    rows = []
    diag = []
    for wid in wells:
        try:
            hz, tw = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        for frac in masks:
            if round(len(hz) * frac) < 5 or (len(hz) - round(len(hz) * frac)) < 10:
                continue
            for name, fn in [
                ("floor", lambda: rung_floor(hz, frac)),  # noqa: B023
                ("geometry2", lambda: rung_geometry(hz, frac, degree=2)),  # noqa: B023
                ("typewell", lambda: rung_typewell_match(hz, tw, frac)),  # noqa: B023
                ("oracle", lambda: rung_oracle(hz, tw, frac)),  # noqa: B023
            ]:
                try:
                    p, t, _ = fn()
                    rows.append({"well": wid, "rung": name, "mask": frac, "rmse": rmse(p, t)})
                except Exception:
                    rows.append({"well": wid, "rung": name, "mask": frac, "rmse": np.nan})
            diag.append({"well": wid, "mask": frac, "geom_r2": geometry_r2(hz, frac)})
    return pd.DataFrame(rows), pd.DataFrame(diag)


# Start with a subset for speed; raise n_wells (or None) for the full run.
N_WELLS = None
results, diag = score_all(WELLS, MASKS, n_wells=N_WELLS)
print(f"scored {results['well'].nunique()} wells")

scored 773 wells


In [8]:
# Per-rung mean RMSE at each mask (the headline table).
pivot = results.groupby(["rung", "mask"])["rmse"].mean().unstack("mask")
order = ["floor", "geometry2", "typewell", "oracle"]
pivot = pivot.reindex([r for r in order if r in pivot.index])
print("Mean RMSE on TVT (ft) by rung x mask — lower is better:")
print(pivot.round(3))

Mean RMSE on TVT (ft) by rung x mask — lower is better:
mask         0.40     0.60     0.73
rung                               
floor       9.924   11.299   13.423
geometry2  37.262  106.917  273.050
typewell   25.960   36.674   54.599
oracle      0.139    0.139    0.139


In [9]:
# Headline diagnostic at the REAL mask: does anything beat the floor, and how far
# is the best real predictor from the oracle?
real = results[np.isclose(results["mask"], 0.73)]
tab = real.groupby("rung")["rmse"].mean()
print("At the REAL 73% mask:")
print(tab.round(3))
if {"floor", "typewell"}.issubset(tab.index):
    print(
        f"\ntypewell - floor : {tab['typewell'] - tab['floor']:+.3f} ft "
        f"({'matcher wins' if tab['typewell'] < tab['floor'] else 'matcher still loses — iterate rung 2'})"
    )
    print(f"typewell - oracle: {tab['typewell'] - tab['oracle']:+.3f} ft  (headroom remaining)")

At the REAL 73% mask:
rung
floor         13.423
geometry2    273.050
oracle         0.139
typewell      54.599
Name: rmse, dtype: float64

typewell - floor : +41.176 ft (matcher still loses — iterate rung 2)
typewell - oracle: +54.461 ft  (headroom remaining)


In [10]:
# Where geometry is weak (low R²), the matcher matters most. Show the split.
d = diag[np.isclose(diag["mask"], 0.73)].merge(
    real[real.rung == "typewell"][["well", "rmse"]].rename(columns={"rmse": "tw_rmse"}),
    on="well",
    how="left",
)
d = d.merge(
    real[real.rung == "floor"][["well", "rmse"]].rename(columns={"rmse": "floor_rmse"}), on="well"
)
d["geom_weak"] = d["geom_r2"] < 0.5
print("At 73%, split by geometry strength:")
print(d.groupby("geom_weak")[["floor_rmse", "tw_rmse"]].mean().round(3))
print("\n(geom_weak=True wells are where the typewell must carry the prediction)")

At 73%, split by geometry strength:
           floor_rmse  tw_rmse
geom_weak                     
False          13.328   53.567
True           14.628   67.564

(geom_weak=True wells are where the typewell must carry the prediction)


## Diagnostics — why the matcher fails, and where to fix it

The matcher loses to the floor at 73%. These cells decompose *why*, so v2 targets
the real cause rather than guessing. Four questions:
1. Is the failure in the **prior** (search centered wrong) or the **pattern matcher**?
2. How often does the eval TVT fall **inside the known range** (is the lateral its own reference)?
3. **Which prior** best constrains the search, and is the best prior well-dependent?
4. How **smooth** is the eval zone (does a continuity constraint help)?

All run on the existing `load_pair`, `tail_mask`, `rmse`, `list_train_wells`,
`REAL_EVAL_FRAC` helpers. Each returns a per-well DataFrame; aggregates print at the end.

In [11]:
def diag_prior_vs_match(wells, frac=0.73, n=None, win_ft=8.0):
    """Separate prior error from pattern-matcher error.
    match_perfect_prior: matcher searching +-10ft of TRUTH (isolates matcher skill).
    match_geom_prior:    matcher searching +-30ft of the geom-deg2 prior (realistic).
    If perfect-prior is good but geom-prior is bad -> fix the PRIOR.
    If perfect-prior is also bad -> fix the PATTERN MATCHER."""
    rows = []
    for wid in wells[:n] if n else wells:
        try:
            hz, tw = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        tvt = hz["TVT"].values.astype(float)
        Z = hz["Z"].values.astype(float)
        MD = hz["MD"].values.astype(float)
        GRz = hz["GR_z"].values.astype(float)
        offset = tvt + Z
        m = tail_mask(len(hz), frac)
        kn = np.where(~m)[0]
        ev = np.where(m)[0]
        if len(kn) < 10 or len(ev) < 5:
            continue
        twt = tw["TVT"].values.astype(float)
        twg = tw["GR_z"].values.astype(float)
        grid = np.arange(-win_ft, win_ft + 1e-9, 0.5)
        order = np.argsort(tvt)
        lt, lg = tvt[order], GRz[order]
        g = ~np.isnan(lg)
        lt, lg = lt[g], lg[g]

        def match(prior, search_ft):
            out = []
            for k, i in enumerate(ev):  # noqa: B023
                exp = prior[k]
                if np.isnan(GRz[i]) or len(lt) < 5:  # noqa: B023
                    out.append(exp)
                    continue
                lp = np.interp(exp + grid, lt, lg)  # noqa: B023
                if np.std(lp) < 1e-6:
                    out.append(exp)
                    continue
                cand = twt[(twt >= exp - search_ft) & (twt <= exp + search_ft)]  # noqa: B023
                bc, bt = -2.0, exp
                for c0 in cand:
                    seg = np.interp(c0 + grid, twt, twg)  # noqa: B023
                    if np.std(seg) < 1e-6:
                        continue
                    cc = np.corrcoef(lp, seg)[0, 1]
                    if cc > bc:
                        bc, bt = cc, c0
                out.append(bt)
            return np.array(out)

        A2 = np.polyfit(MD[kn], offset[kn], 2)
        geo_prior = np.polyval(A2, MD[ev]) - Z[ev]
        rows.append(
            {
                "well": wid,
                "prior_geom2_rmse": rmse(geo_prior, tvt[ev]),
                "match_perfect_prior": rmse(match(tvt[ev], 10), tvt[ev]),
                "match_geom_prior": rmse(match(geo_prior, 30), tvt[ev]),
            }
        )
    return pd.DataFrame(rows)


d1 = diag_prior_vs_match(WELLS, frac=REAL_EVAL_FRAC, n=40)
print(d1.describe().loc[["mean", "50%"]].round(2))
print("\nInterpretation:")
print("  match_perfect_prior << match_geom_prior  =>  the PRIOR is the problem")
print("  match_perfect_prior also large           =>  the PATTERN MATCHER needs work")

      prior_geom2_rmse  match_perfect_prior  match_geom_prior
mean            230.47                 2.13            231.78
50%             177.47                 1.49            177.09

Interpretation:
  match_perfect_prior << match_geom_prior  =>  the PRIOR is the problem
  match_perfect_prior also large           =>  the PATTERN MATCHER needs work


In [12]:
def diag_eval_inside(wells, frac=0.73, n=None):
    """How much of the eval tail falls inside the KNOWN TVT range (=> the known
    lateral is itself a usable reference), and how smooth the eval zone is."""
    rows = []
    for wid in wells[:n] if n else wells:
        try:
            hz, _ = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        tvt = hz["TVT"].values.astype(float)
        m = tail_mask(len(hz), frac)
        kn = np.where(~m)[0]
        ev = np.where(m)[0]
        if len(kn) < 10 or len(ev) < 5:
            continue
        kmin, kmax = tvt[kn].min(), tvt[kn].max()
        rows.append(
            {
                "well": wid,
                "frac_eval_inside_known": float(((tvt[ev] >= kmin) & (tvt[ev] <= kmax)).mean()),
                "eval_dtvt_abs_mean": float(np.abs(np.diff(tvt[ev])).mean()),
                "eval_tvt_span": float(tvt[ev].max() - tvt[ev].min()),
            }
        )
    return pd.DataFrame(rows)


d2 = diag_eval_inside(WELLS, frac=REAL_EVAL_FRAC, n=40)
print(d2.describe().loc[["mean", "50%", "min", "max"]].round(3))
print(
    f"\nwells with >90% eval inside known range: {(d2['frac_eval_inside_known'] > 0.9).mean():.0%}"
)
print("  high => a 'self-reference' matcher (use the known lateral, not just typewell) is viable")

      frac_eval_inside_known  eval_dtvt_abs_mean  eval_tvt_span
mean                   0.585               0.018         29.083
50%                    0.614               0.018         25.990
min                    0.000               0.009          9.940
max                    1.000               0.029         72.250

wells with >90% eval inside known range: 25%
  high => a 'self-reference' matcher (use the known lateral, not just typewell) is viable


In [13]:
def diag_prior_sweep(wells, frac=0.73, n=None):
    """Compare candidate priors per well. The winner tells matcher v2 where to
    center its search; if the winner varies across wells, v2 must choose per well."""
    rows = []
    for wid in wells[:n] if n else wells:
        try:
            hz, _ = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        tvt = hz["TVT"].values.astype(float)
        Z = hz["Z"].values.astype(float)
        MD = hz["MD"].values.astype(float)
        offset = tvt + Z
        m = tail_mask(len(hz), frac)
        kn = np.where(~m)[0]
        ev = np.where(m)[0]
        if len(kn) < 10 or len(ev) < 5:
            continue
        A1 = np.polyfit(MD[kn], offset[kn], 1)
        A2 = np.polyfit(MD[kn], offset[kn], 2)
        rows.append(
            {
                "well": wid,
                "flat_tvt": rmse(tvt[kn[-1]], tvt[ev]),
                "flat_offset": rmse(offset[kn[-1]] - Z[ev], tvt[ev]),
                "geom1": rmse(np.polyval(A1, MD[ev]) - Z[ev], tvt[ev]),
                "geom2": rmse(np.polyval(A2, MD[ev]) - Z[ev], tvt[ev]),
            }
        )
    df = pd.DataFrame(rows)
    if len(df):
        cols = ["flat_tvt", "flat_offset", "geom1", "geom2"]
        df["best_prior"] = df[cols].idxmin(axis=1)
    return df


d3 = diag_prior_sweep(WELLS, frac=REAL_EVAL_FRAC, n=40)
print("mean RMSE per prior:")
print(d3[["flat_tvt", "flat_offset", "geom1", "geom2"]].mean().round(2))
print("\nbest prior per well (count):")
print(d3["best_prior"].value_counts())
print("  if this is split across priors, v2 must SELECT the prior per well")

mean RMSE per prior:
flat_tvt        12.24
flat_offset     91.59
geom1           49.28
geom2          230.47
dtype: float64

best prior per well (count):
best_prior
flat_tvt    38
geom1        2
Name: count, dtype: int64
  if this is split across priors, v2 must SELECT the prior per well


In [14]:
def diag_search_width(wells, frac=0.73, n=None, win_ft=8.0, conf_thr=0.5):
    """For each well: floor (flat-TVT prior) vs confidence-gated matcher searching
    around that prior at several widths. The matcher refines toward the typewell
    only when best-corr >= conf_thr, else holds the prior (avoids wandering on
    flat wells). Tells us if gated search beats the floor, and at what width."""
    widths = [5, 10, 15, 25]
    rows = []
    for wid in wells[:n] if n else wells:
        try:
            hz, tw = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        tvt = hz["TVT"].values.astype(float)
        GRz = hz["GR_z"].values.astype(float)
        m = tail_mask(len(hz), frac)
        kn = np.where(~m)[0]
        ev = np.where(m)[0]
        if len(kn) < 10 or len(ev) < 5:
            continue
        twt = tw["TVT"].values.astype(float)
        twg = tw["GR_z"].values.astype(float)
        prior = tvt[kn[-1]]
        grid = np.arange(-win_ft, win_ft + 1e-9, 0.5)
        order = np.argsort(tvt)
        lt, lg = tvt[order], GRz[order]
        g = ~np.isnan(lg)
        lt, lg = lt[g], lg[g]
        rec = {
            "well": wid,
            "eval_span": float(tvt[ev].max() - tvt[ev].min()),
            "floor": rmse(prior, tvt[ev]),
        }
        for sft in widths:
            out = []
            for i in ev:
                if np.isnan(GRz[i]) or len(lt) < 5:
                    out.append(prior)
                    continue
                lp = np.interp(prior + grid, lt, lg)
                if np.std(lp) < 1e-6:
                    out.append(prior)
                    continue
                cand = twt[(twt >= prior - sft) & (twt <= prior + sft)]
                bc, bt = -2.0, prior
                for c0 in cand:
                    seg = np.interp(c0 + grid, twt, twg)
                    if np.std(seg) < 1e-6:
                        continue
                    cc = np.corrcoef(lp, seg)[0, 1]
                    if cc > bc:
                        bc, bt = cc, c0
                out.append(bt if bc >= conf_thr else prior)
            rec[f"gated_search{sft}"] = rmse(np.array(out), tvt[ev])
        rows.append(rec)
    return pd.DataFrame(rows)


d4 = diag_search_width(WELLS, frac=REAL_EVAL_FRAC, n=40)
print("mean RMSE (gated matcher vs floor):")
print(d4.drop(columns=["well"]).mean().round(2))
# the decisive split: on high-movement wells, does gated search beat the floor?
hi = d4[d4["eval_span"] > 15]
print(f"\nhigh-movement wells (eval_span>15ft, n={len(hi)}):")
if len(hi):
    print(hi[["floor", "gated_search5", "gated_search10", "gated_search15"]].mean().round(2))
print(
    "  if gated_search < floor here, THIS is matcher v2: flat prior + confidence-gated typewell refine"
)

mean RMSE (gated matcher vs floor):
eval_span         29.08
floor             12.24
gated_search5     12.40
gated_search10    12.62
gated_search15    12.60
gated_search25    14.92
dtype: float64

high-movement wells (eval_span>15ft, n=37):
floor             12.71
gated_search5     12.86
gated_search10    13.09
gated_search15    13.07
dtype: float64
  if gated_search < floor here, THIS is matcher v2: flat prior + confidence-gated typewell refine


In [15]:
def method_sweep(wells, frac=0.73, n=None):
    """Run many localization methods per well at the real mask. Returns per-well
    RMSE for each method + floor + oracle + eval_span. Split by eval_span when
    reading: flat wells (small span) make the floor look unbeatable trivially;
    the high-span wells are where a real method must prove itself."""
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.isotonic import IsotonicRegression

    try:
        from dtaidistance import dtw

        HAVE_DTW = True
    except Exception:
        HAVE_DTW = False
    rows = []
    for wid in wells[:n] if n else wells:
        try:
            hz, tw = load_pair(wid)
        except Exception:
            continue
        if "TVT" not in hz or hz["TVT"].isna().all():
            continue
        tvt = hz["TVT"].values.astype(float)
        Z = hz["Z"].values.astype(float)
        MD = hz["MD"].values.astype(float)
        GRz = hz["GR_z"].values.astype(float)
        m = tail_mask(len(hz), frac)
        kn = np.where(~m)[0]
        ev = np.where(m)[0]
        if len(kn) < 20 or len(ev) < 20:
            continue
        twt = tw["TVT"].values.astype(float)
        twg = tw["GR_z"].values.astype(float)
        prior = tvt[kn[-1]]
        tru = tvt[ev]
        r = {
            "well": wid,
            "eval_span": float(tru.max() - tru.min()),
            "floor": rmse(prior, tru),
            "oracle": rmse(np.array([twt[np.argmin(np.abs(twt - x))] for x in tru]), tru),
        }

        # nearest-GR constrained
        def nearest(s):
            o = []
            for i in ev:
                g = GRz[i]
                if np.isnan(g):
                    o.append(prior)
                    continue
                c = np.where(np.abs(twt - prior) <= s)[0]
                o.append(twt[c[np.argmin(np.abs(twg[c] - g))]] if len(c) else prior)
            return np.array(o)

        r["nearGR_s30"] = rmse(nearest(30), tru)
        # isotonic GR->TVT
        try:
            iso = IsotonicRegression(out_of_bounds="clip").fit(twg, twt)
            r["isotonic"] = rmse(
                iso.predict(np.nan_to_num(GRz[ev], nan=np.nanmedian(GRz[kn]))), tru
            )
        except Exception:
            r["isotonic"] = np.nan
        # HGB on relative features (no absolute coords)
        try:
            dZ = np.gradient(Z)
            dMD = np.gradient(MD)
            incl = dZ / np.where(dMD == 0, 1, dMD)

            def F(idx):
                return np.c_[np.nan_to_num(GRz[idx]), incl[idx], Z[idx] - Z[kn[-1]]]

            hgb = HistGradientBoostingRegressor(max_iter=150).fit(F(kn), tvt[kn])
            r["hgb_rel"] = rmse(hgb.predict(F(ev)), tru)
        except Exception:
            r["hgb_rel"] = np.nan
        # TVT from known Z (no MD extrapolation)
        try:
            A = np.polyfit(Z[kn], tvt[kn], 1)
            r["tvt_from_Z"] = rmse(np.polyval(A, Z[ev]), tru)
        except Exception:
            r["tvt_from_Z"] = np.nan
        # DTW banded
        if HAVE_DTW:
            try:
                band = np.where(np.abs(twt - prior) <= 40)[0]
                if len(band) > 10:
                    from collections import defaultdict

                    path = dtw.warping_path(
                        np.nan_to_num(GRz[ev], nan=np.nanmedian(GRz[kn])), twg[band]
                    )
                    mp = defaultdict(list)
                    for li, ti in path:
                        mp[li].append(twt[band][ti])
                    dp = np.array([np.mean(mp[k]) if k in mp else prior for k in range(len(ev))])
                    r["dtw_band"] = rmse(dp, tru)
            except Exception:
                r["dtw_band"] = np.nan
        rows.append(r)
    return pd.DataFrame(rows)


sweep = method_sweep(WELLS, frac=REAL_EVAL_FRAC, n=None)  # n=None = all wells
method_cols = [c for c in sweep.columns if c not in ("well", "eval_span")]

print("=== ALL WELLS: mean RMSE ===")
print(sweep[method_cols].mean().sort_values().round(2))

# THE decisive view: split by movement. On high-span wells the floor is beatable.
for lo, hi, label in [
    (0, 5, "FLAT (span<5ft)"),
    (5, 15, "MED (5-15ft)"),
    (15, 999, "HIGH (>15ft)"),
]:
    sub = sweep[(sweep.eval_span >= lo) & (sweep.eval_span < hi)]
    if len(sub):
        print(f"\n=== {label}: n={len(sub)} ===")
        mt = sub[method_cols].mean().sort_values()
        floor_v = sub["floor"].mean()
        for k, v in mt.items():
            star = " <<<BEATS FLOOR" if (v < floor_v and k not in ("floor", "oracle")) else ""
            print(f"  {k:14s} {v:8.2f}{star}")

=== ALL WELLS: mean RMSE ===
oracle          0.14
floor          13.42
nearGR_s30     22.75
hgb_rel        49.90
tvt_from_Z    110.43
isotonic      282.56
dtype: float64

=== MED (5-15ft): n=64 ===
  oracle             0.13
  floor              4.97
  nearGR_s30        18.79
  hgb_rel           39.01
  tvt_from_Z       100.82
  isotonic         248.60

=== HIGH (>15ft): n=709 ===
  oracle             0.14
  floor             14.19
  nearGR_s30        23.11
  hgb_rel           50.88
  tvt_from_Z       111.30
  isotonic         285.63


## Where to iterate (rung 2 is the open problem)

The harness above is correct and the floor/geometry/oracle rungs are solid. The
**typewell matcher is the scaffold to improve**. Concrete next moves, roughly in
order of expected payoff:

1. **Better prior than degree-1 geometry** inside the matcher — at 73% the prior is
   poor, so the search window must be wide, which re-introduces ambiguity. A robust
   prior (e.g. median offset of the last known segment) may localize better.
2. **Pattern features instead of single-window correlation** — multi-scale windows,
   GR gradient/shape, or DTW to handle the lateral's variable TVT-resolution.
3. **Learned residual correction** — once the matcher gives `match_tvt` and a
   confidence (best-corr), train a model to predict the *offset residual* from
   `[match_tvt, corr, geom_prior, GR shape stats]`, scored on the pad-CV harness
   from notebook 3. Never let it predict absolute offset (trees can't extrapolate).
4. **Use `Geology_canon`** — restrict the typewell search to the formation the
   lateral is likely in, cutting ambiguity.

Score every change here, at the **73% mask**, against `floor` and `oracle`. The
target is to close the floor→oracle gap; any rung that does not beat `floor` at 73%
is not yet helping.